<a href="https://colab.research.google.com/github/tmtngu/OANNANTMT/blob/main/GREENMOVE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyngrok streamlit geopy osmnx networkx folium streamlit-folium scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 47.6 MB/s eta 0:00:00


In [2]:
import getpass
from pyngrok import ngrok

ngrok_key = getpass.getpass("Nhập ngrok Authtoken của bạn vào đây: ")
ngrok.set_auth_token(ngrok_key)

Nhập ngrok Authtoken của bạn vào đây: ··········


In [64]:
%%writefile app.py
import streamlit as st
from geopy.geocoders import Nominatim
import requests
import folium
from streamlit_folium import st_folium
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import base64
import streamlit as st

# --- HÀM TẢI VÀ MÃ HÓA ẢNH TỪ LINK ---
def get_base64_from_url(url):
    response = requests.get(url)
    if response.status_code == 200:
        return base64.b64encode(response.content).decode()
    else:
        return None

# --- 1. CÀI ĐẶT BACKGROUND ---
# Thay link ảnh m muốn vào đây
img_url = "https://i.ibb.co/sdWgKzDs/Gemini-Generated-Image-vel3h7vel3h7vel3-Picsart-Ai-Image-Enhancer.png"
bin_str = get_base64_from_url(img_url)

if bin_str:
    st.markdown(f"""
        <style>
        .stApp {{
            background-image: url("data:image/png;base64,{bin_str}");
            background-size: cover;
            background-position: center;
            background-attachment: fixed;
        }}

        /* Chỉnh các ô Input & Selectbox để hiện chữ đen trên nền trắng */
        div[data-baseweb="select"] div, input {{
            color: #000000 !important;
            -webkit-text-fill-color: #000000 !important;
            font-weight: 500 !important;
        }}

        div[data-baseweb="base-input"] {{
            background-color: #FFFFFF !important;
            border-radius: 8px !important;
        }}
        </style>
    """, unsafe_allow_html=True)

# --- 1. HỆ THỐNG FUZZY LOGIC ĐỊNH GIÁ ĐỘNG (12 LUẬT ĐẦY ĐỦ) ---
@st.cache_resource
def setup_fuzzy_pricing():
    thoi_tiet_fz = ctrl.Antecedent(np.arange(0, 11, 1), 'thoi_tiet')
    giao_thong_fz = ctrl.Antecedent(np.arange(0, 11, 1), 'giao_thong')
    loai_xe_fz = ctrl.Antecedent(np.arange(0, 11, 1), 'loai_xe')
    he_so_gia = ctrl.Consequent(np.arange(1.0, 2.6, 0.1), 'he_so_gia')

    thoi_tiet_fz['nang'] = fuzz.trimf(thoi_tiet_fz.universe, [0, 0, 5])
    thoi_tiet_fz['mua'] = fuzz.trimf(thoi_tiet_fz.universe, [0, 5, 10])
    thoi_tiet_fz['bao'] = fuzz.trimf(thoi_tiet_fz.universe, [5, 10, 10])

    giao_thong_fz['binh_thuong'] = fuzz.trimf(giao_thong_fz.universe, [0, 0, 10])
    giao_thong_fz['cao_diem'] = fuzz.trimf(giao_thong_fz.universe, [0, 10, 10])

    loai_xe_fz['bike'] = fuzz.trimf(loai_xe_fz.universe, [0, 0, 10])
    loai_xe_fz['car'] = fuzz.trimf(loai_xe_fz.universe, [0, 10, 10])

    he_so_gia['binh_thuong'] = fuzz.trimf(he_so_gia.universe, [1.0, 1.0, 1.3])
    he_so_gia['tang_nhe'] = fuzz.trimf(he_so_gia.universe, [1.2, 1.5, 1.8])
    he_so_gia['tang_manh'] = fuzz.trimf(he_so_gia.universe, [1.8, 2.0, 2.5])

    rule1 = ctrl.Rule(thoi_tiet_fz['nang'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['bike'], he_so_gia['binh_thuong'])
    rule2 = ctrl.Rule(thoi_tiet_fz['nang'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['car'], he_so_gia['binh_thuong'])
    rule3 = ctrl.Rule(thoi_tiet_fz['nang'] & giao_thong_fz['cao_diem'] & loai_xe_fz['bike'], he_so_gia['tang_nhe'])
    rule4 = ctrl.Rule(thoi_tiet_fz['nang'] & giao_thong_fz['cao_diem'] & loai_xe_fz['car'], he_so_gia['tang_nhe'])
    rule5 = ctrl.Rule(thoi_tiet_fz['mua'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['bike'], he_so_gia['tang_nhe'])
    rule6 = ctrl.Rule(thoi_tiet_fz['mua'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['car'], he_so_gia['tang_nhe'])
    rule7 = ctrl.Rule(thoi_tiet_fz['mua'] & giao_thong_fz['cao_diem'] & loai_xe_fz['bike'], he_so_gia['tang_nhe'])
    rule8 = ctrl.Rule(thoi_tiet_fz['mua'] & giao_thong_fz['cao_diem'] & loai_xe_fz['car'], he_so_gia['tang_manh'])
    rule9 = ctrl.Rule(thoi_tiet_fz['bao'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['bike'], he_so_gia['tang_nhe'])
    rule10 = ctrl.Rule(thoi_tiet_fz['bao'] & giao_thong_fz['binh_thuong'] & loai_xe_fz['car'], he_so_gia['tang_manh'])
    rule11 = ctrl.Rule(thoi_tiet_fz['bao'] & giao_thong_fz['cao_diem'] & loai_xe_fz['bike'], he_so_gia['tang_manh'])
    rule12 = ctrl.Rule(thoi_tiet_fz['bao'] & giao_thong_fz['cao_diem'] & loai_xe_fz['car'], he_so_gia['tang_manh'])

    pricing_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10, rule11, rule12])
    return ctrl.ControlSystemSimulation(pricing_ctrl)

pricing_sim = setup_fuzzy_pricing()

# --- 2. CẤU HÌNH GIAO DIỆN & STYLE ---
st.set_page_config(page_title="Green Move", layout="wide", page_icon="🌿")

st.markdown("""
    <style>
    @import url('https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css');
    .stApp { background-color: #F4FAF5; color: #000000; font-family: 'Segoe UI', sans-serif; }

    .splash-container { position: fixed; top: 0; left: 0; width: 100vw; height: 100vh; background-color: #E8F2E3; z-index: 999999; display: flex; flex-direction: column; justify-content: center; align-items: center; animation: hideSplash 0.8s ease-in-out forwards; animation-delay: 2.5s; pointer-events: none; }
    .splash-icon { font-size: 250px; color: #1A4A28; margin-bottom: 20px; opacity: 0; animation: scaleInSplash 1.2s cubic-bezier(0.175, 0.885, 0.32, 1.275) forwards; }
    .splash-text { color: #1A4A28; font-family: 'Impact', sans-serif; font-size: 70px; opacity: 0; animation: zoomInText 1s forwards; animation-delay: 0.5s; }

    @keyframes scaleInSplash { 0% { transform: scale(0.2) translateY(-100px); opacity: 0; } 100% { transform: scale(1) translateY(0); opacity: 1; } }
    @keyframes zoomInText { 0% { transform: scale(0.3); opacity: 0; } 100% { transform: scale(1); opacity: 1; } }
    @keyframes hideSplash { 0% { opacity: 1; visibility: visible; } 100% { opacity: 0; visibility: hidden; } }

    .main-title { color: #1A4A28; font-family: 'Impact', sans-serif; font-size: 48px; text-transform: uppercase; margin-bottom: 30px; text-align: center; }
    div[data-testid="column"]:nth-of-type(1), div[data-testid="column"]:nth-of-type(3) { background-color: #E8F2E3; padding: 25px; border-radius: 20px; box-shadow: 0px 4px 10px rgba(26, 74, 40, 0.08); border: 1px solid #CDE0C4; }
    div[data-baseweb="input"], div[data-baseweb="select"] > div { background-color: #FFFFFF !important; border-radius: 15px !important; border: 2px solid #1A4A28 !important; }
    div[data-testid="stTextInput"] label p, div[data-testid="stSelectbox"] label p { display: none; }
    .pill-label { background-color: #A3C096; color: #000000; padding: 6px 15px; border-radius: 15px; font-weight: bold; display: inline-flex; align-items: center; gap: 8px; margin-bottom: 8px; font-size: 15px; }
    .pill-label i { color: #1A4A28; }
    .info-box { background-color: #FFFFFF; border-radius: 15px; padding: 12px; margin-bottom: 10px; border: 1px dashed #1A4A28; color: #000; }
    .result-box { background-color: #FFFFFF; border: 2px solid #1A4A28; border-radius: 15px; padding: 15px; margin-top: 20px; font-weight: 500; line-height: 1.6; }

    button[kind="secondary"] { background-color: #A3C096 !important; color: #000000 !important; font-weight: bold !important; font-size: 16px !important; border-radius: 15px !important; padding: 10px !important; width: 100% !important; border: 1px solid #1A4A28 !important; margin-bottom: 10px !important; transition: all 0.3s !important; }
    button[kind="secondary"]:hover { background-color: #8EB080 !important; transform: translateY(-2px) !important; box-shadow: 0px 4px 8px rgba(0,0,0,0.15) !important; }
    button[kind="primary"] { background-color: #1A4A28 !important; color: white !important; font-weight: bold !important; font-size: 22px !important; border-radius: 30px !important; padding: 15px 30px !important; width: 100% !important; border: none !important; box-shadow: 0px 5px 15px rgba(0,0,0,0.2) !important; transition: all 0.3s !important; }
    button[kind="primary"]:hover { background-color: #266b3a !important; transform: scale(1.05) !important; box-shadow: 0px 8px 20px rgba(0,0,0,0.3) !important; }
    </style>

    <div class="splash-container"><i class="fa-solid fa-leaf splash-icon"></i><div class="splash-text">GREEN MOVE</div></div>
""", unsafe_allow_html=True)

# --- 3. HÀM TÌM ĐƯỜNG ---
@st.cache_data(show_spinner=False)
def get_info(address):
    geolocator = Nominatim(user_agent="ueh_green_v3", timeout=10)
    try:
        loc = geolocator.geocode(address)
        if loc: return {"addr": loc.address, "lat": loc.latitude, "lon": loc.longitude}
    except: pass
    return None

def get_route(latA, lonA, latB, lonB):
    url = f"http://router.project-osrm.org/route/v1/driving/{lonA},{latA};{lonB},{latB}?overview=full&geometries=geojson"
    try:
        res = requests.get(url, timeout=5).json()
        if res.get("code") == "Ok":
            dist = res["routes"][0]["distance"] / 1000
            coords = [(l[1], l[0]) for l in res["routes"][0]["geometry"]["coordinates"]]
            return coords, dist
    except: pass
    return None, None

# --- KHỞI TẠO BIẾN SESSION STATE ---
if 'den_pos' not in st.session_state: st.session_state.den_pos = ""
if 'don_pos' not in st.session_state: st.session_state.don_pos = ""
if 'show_result' not in st.session_state: st.session_state.show_result = False
if 'html_bill' not in st.session_state: st.session_state.html_bill = ""
if 'route_path' not in st.session_state: st.session_state.route_path = []
if 'loc_a' not in st.session_state: st.session_state.loc_a = None
if 'loc_b' not in st.session_state: st.session_state.loc_b = None

st.markdown("<div class='main-title'><i class='fa-solid fa-leaf'></i> GREEN MOVE</div>", unsafe_allow_html=True)

# --- 4. BỐ CỤC GIAO DIỆN ---
st.markdown(f"""
    <style>
    /* 1. GẮN BACKGROUND PHỦ KÍN */
    .stApp, [data-testid="stSidebar"] {{
        background-image: url("data:image/png;base64,{bin_str}");
        background-size: cover;
        background-position: center;
        background-repeat: no-repeat;
        background-attachment: fixed;
    }}

    [data-testid="stSidebar"] > div:first-child {{
        background-color: rgba(255, 255, 255, 0.1);
    }}

    /* 2. CHỈNH Ô SELECTBOX */
    div[data-baseweb="select"] > div {{
        background-color: #FFFFFF !important;
        color: #000000 !important;
        -webkit-text-fill-color: #000000 !important;
        font-weight: 500 !important;
        border-radius: 8px !important;
    }}

    /* 3. CHỈNH Ô INPUT */
    div[data-baseweb="base-input"] {{
        background-color: #FFFFFF !important;
        border-radius: 8px !important;
        border: 1px solid #d1e7dd !important;
    }}

    input {{
        color: #000000 !important;
        -webkit-text-fill-color: #000000 !important;
        font-weight: 500 !important;
    }}

   /* 4. CHỈNH MÀU CAM CHO NÚT XÁC NHẬN */

    .stButton > button {{

        background-color: #FFFFFF !important; /* Màu cam */

        color: white !important;

        border: none !important;

        border-radius: 8px !important;

        height: 42px !important;

        width: 100% !important;

    }}



    .stButton > button:hover {{

        background-color: #28a745 !important;

    }}

    /* 5. FIX CHỮ VÀ PLACEHOLDER */
    input::placeholder {{
        color: #666666 !important;
    }}

    div[data-testid="stMarkdownContainer"] p {{
        color: #000000 !important;
        margin-bottom: 0px !important;
    }}
    </style>
""", unsafe_allow_html=True)

col_left, col_spacing, col_right = st.columns([1, 0.05, 1])

with col_left:
    # PHẦN ĐIỂM ĐÓN
    st.markdown("<div class='pill-label'><i class='fa-solid fa-map-pin'></i> Nhập điểm đón</div>", unsafe_allow_html=True)
    c_don_in, c_don_btn = st.columns([0.7, 0.3]) # Dồn nút qua phải
    with c_don_in:
        diem_don = st.text_input("don_in", placeholder="Ví dụ: Cơ sở A UEH", label_visibility="collapsed")
    with c_don_btn:
        if st.button("👉 Xác nhận", key="btn_don"):
            info = get_info(diem_don)
            st.session_state.don_pos = info['addr'] if info else "Không tìm thấy!"
    if st.session_state.don_pos: st.markdown(f"<div class='info-box'>{st.session_state.don_pos}</div>", unsafe_allow_html=True)

    # PHẦN ĐIỂM ĐẾN
    st.markdown("<div class='pill-label'><i class='fa-solid fa-flag-checkered'></i> Nhập điểm đến</div>", unsafe_allow_html=True)
    c_den_in, c_den_btn = st.columns([0.7, 0.3]) # Dồn nút qua phải
    with c_den_in:
        diem_den = st.text_input("den_in", placeholder="Ví dụ: Dinh Độc Lập", label_visibility="collapsed")
    with c_den_btn:
        if st.button("👉 Xác nhận", key="btn_den"):
            info = get_info(diem_den)
            st.session_state.den_pos = info['addr'] if info else "Không tìm thấy!"
    if st.session_state.den_pos: st.markdown(f"<div class='info-box'>{st.session_state.den_pos}</div>", unsafe_allow_html=True)

with col_right:
    c1, c2 = st.columns(2)
    with c1:
        st.markdown("<div class='pill-label'><i class='fa-solid fa-cloud-sun'></i> Thời tiết</div>", unsafe_allow_html=True)
        thoi_tiet = st.selectbox("tt", ["☀️ Nắng", "🌧️ Mưa", "⛈️ Bão"], label_visibility="collapsed")
    with c2:
        st.markdown("<div class='pill-label'><i class='fa-solid fa-bolt'></i> Giờ cao điểm</div>", unsafe_allow_html=True)
        gio_cao_diem = st.selectbox("gcd", ["Không", "⚡ Có"], label_visibility="collapsed")

    c3, c4 = st.columns(2)
    with c3:
        st.markdown("<div class='pill-label'><i class='fa-solid fa-car'></i> Loại xe</div>", unsafe_allow_html=True)
        loai_xe = st.selectbox("lx", ["🏍️ Green-Bike", "🚗 Green-Car"], label_visibility="collapsed")
    with c4:
        st.markdown("<div class='pill-label'><i class='fa-solid fa-credit-card'></i> Thanh toán</div>", unsafe_allow_html=True)
        thanh_toan = st.selectbox("pay", ["📱 Momo", "💳 ZaloPay", "💵 Tiền Mặt"], label_visibility="collapsed")

    st.markdown("<div class='pill-label'><i class='fa-solid fa-comment-dots'></i> Ghi chú cho tài xế</div>", unsafe_allow_html=True)
    ghi_chu = st.text_input("gc_in", placeholder="Ví dụ: Cổng sau, gọi trước khi đến...", label_visibility="collapsed")
# --- 5. NÚT ĐẶT XE VÀ XỬ LÝ ---
st.write("")
col_b1, col_b2, col_b3 = st.columns([1, 1.5, 1])

with col_b2:
    if st.button("🚕 ĐẶT XE NGAY", type="primary", use_container_width=True):
        if not diem_don or not diem_den:
            st.error("⚠️ Vui lòng nhập và xác nhận địa chỉ!")
        else:
            with st.spinner("Đang tính toán hành trình..."):
                locA, locB = get_info(diem_don), get_info(diem_den)
                if locA and locB:
                    path, km = get_route(locA['lat'], locA['lon'], locB['lat'], locB['lon'])
                    if km:
                        # Logic Mờ
                        tt_val = 0 if "Nắng" in thoi_tiet else (5 if "Mưa" in thoi_tiet else 10)
                        gt_val = 0 if gio_cao_diem == "Không" else 10
                        lx_val = 0 if "Bike" in loai_xe else 10

                        pricing_sim.input['thoi_tiet'] = tt_val
                        pricing_sim.input['giao_thong'] = gt_val
                        pricing_sim.input['loai_xe'] = lx_val
                        pricing_sim.compute()

                        he_so = pricing_sim.output['he_so_gia']
                        gia_co_ban = km * (6000 if "Bike" in loai_xe else 12000)
                        tong_tien = gia_co_ban * he_so
                        gia_tb_km = tong_tien / km if km > 0 else 0

                        # Cập nhật Session State
                        st.session_state.html_bill = f"""
                        <div class='result-box' style='text-align: center;'>
                            <h3 style='color: #1A4A28;'>🍃 CHI TIẾT CHUYẾN ĐI 🍃</h3>
                            <p><b>Khoảng cách:</b> {km:.2f} km | <b>Thời tiết:</b> {thoi_tiet}</p>
                            <p><b>Dịch vụ:</b> {loai_xe} | <b>Thanh toán:</b> {thanh_toan}</p>
                            <p><b>Ghi chú:</b> {ghi_chu if ghi_chu else "Không có"}</p>
                            <hr style='border: 1px dashed #CDE0C4;'>
                            <p style='color:#555; font-size:14px;'>Hệ số giá động: <b>x{he_so:.2f}</b> | Giá TB/1km: <b>{gia_tb_km:,.0f} VNĐ</b></p>
                            <h2 style='color: #d32f2f;'>TỔNG TIỀN: {tong_tien:,.0f} VNĐ</h2>
                        </div>
                        """
                        st.session_state.route_path = path
                        st.session_state.loc_a = locA
                        st.session_state.loc_b = locB
                        st.session_state.show_result = True
                    else:
                        st.error("⚠️ Lỗi tìm đường!")
                else:
                    st.error("⚠️ Lỗi định vị địa chỉ!")


# --- 6. HIỂN THỊ KẾT QUẢ ---
if st.session_state.show_result:
    st.markdown(st.session_state.html_bill, unsafe_allow_html=True)

    locA = st.session_state.loc_a
    locB = st.session_state.loc_b

    m = folium.Map(location=[(locA['lat']+locB['lat'])/2, (locA['lon']+locB['lon'])/2], zoom_start=13)
    folium.PolyLine(st.session_state.route_path, color='#1A4A28', weight=6).add_to(m)
    folium.Marker([locA['lat'], locA['lon']], icon=folium.Icon(color='green'), tooltip="Điểm đón").add_to(m)
    folium.Marker([locB['lat'], locB['lon']], icon=folium.Icon(color='red'), tooltip="Điểm đến").add_to(m)

    st_folium(m, use_container_width=True, height=400, returned_objects=[])

Overwriting app.py


In [65]:
import subprocess
import time
from pyngrok import ngrok

# 1. Tắt sạch các website cũ đang chạy kẹt
!pkill -f streamlit
ngrok.kill()

# 2. Khởi động ứng dụng mới
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"])

# 3. Đợi vài giây cho web load
time.sleep(4)

# 4. Lấy link mới
public_url = ngrok.connect(8501).public_url
print(f"\n=======================================================")
print(f"✅ HOÀN TẤT! BẤM VÀO LINK DƯỚI ĐỂ MỞ APP GIAO DIỆN MOBILE:")
print(f"👉 {public_url}")
print(f"=======================================================")


✅ HOÀN TẤT! BẤM VÀO LINK DƯỚI ĐỂ MỞ APP GIAO DIỆN MOBILE:
👉 https://griminess-patrol-attribute.ngrok-free.dev
